# 03 — Strategy Selection

The `StrategySelector` is a lightweight decision engine. It takes a `ClassificationResult` (from the classifier) and returns the name of the strategy to execute.

## Strategy options

| Strategy | When to use |
|----------|------------|
| `direct_llm` | Simple questions the LLM can answer from its own knowledge |
| `document_rag` | Factual questions best answered from a curated document store |
| `web_search_rag` | Time‑sensitive questions needing live web data |
| `hybrid_rag` | Complex questions that benefit from both documents *and* web |
| `graph_rag` | Questions about entities and their relationships |
| `self_rag` | Fallback when other strategies are not clearly indicated |

## 1. Import Selector & Classifier

In [ ]:
import sys
sys.path.insert(0, "..")

try:
    from backend.adaptive_rag.router.query_classifier import QueryClassifier
    from backend.adaptive_rag.router.strategy_selector import StrategySelector
    print("✓ Imported from backend")
except ImportError:
    print("⚠ Backend import failed — using fallback")
    from dataclasses import dataclass
    @dataclass
    class ClassificationResult:
        query_type: str = ""
        complexity: str = "simple"
        needs_docs: bool = False
        needs_web: bool = False
        is_time_sensitive: bool = False
        needs_graph: bool = False

    class QueryClassifier:
        def classify(self, q): return ClassificationResult()

    class StrategySelector:
        def select(self, c) -> str:
            if not c.needs_docs and not c.needs_web and not c.needs_graph:
                return "direct_llm"
            if c.needs_docs and not c.needs_web:
                return "document_rag"
            if c.needs_web and not c.needs_docs:
                return "web_search_rag"
            if c.needs_docs and c.needs_web:
                return "hybrid_rag"
            if c.needs_graph:
                return "graph_rag"
            return "self_rag"

classifier = QueryClassifier()
selector = StrategySelector()
print("✓ Ready")

## 2. Selection Logic

The selector uses a priority‑based decision tree:

In [ ]:
queries = [
    "What is the capital of France?",
    "Latest news on AI regulation",
    "Explain the impact of climate change on agriculture using scientific studies",
    "What is the relationship between inflation and interest rates?",
    "What do you think is the best smartphone in 2025?",
    "How to bake a chocolate cake",
]

print(f"{'Query':<65} {'Type':<14} {'Strategy':<18}")
print("─" * 97)
for q in queries:
    c = classifier.classify(q)
    strategy = selector.select(c)
    print(f"{q[:64]:<65} {c.query_type:<14} {strategy:<18}")

## 3. Step Through the Decision Tree

Let's trace *why* a particular strategy is chosen for a given query:

In [ ]:
def trace_selection(query):
    c = classifier.classify(query)
    steps = []
    
    steps.append(f"needs_docs  = {c.needs_docs}")
    steps.append(f"needs_web   = {c.needs_web}")
    steps.append(f"needs_graph = {c.needs_graph}")
    
    if not c.needs_docs and not c.needs_web and not c.needs_graph:
        steps.append("→ All flags False → direct_llm")
    elif c.needs_docs and not c.needs_web:
        steps.append("→ needs_docs only → document_rag")
    elif c.needs_web and not c.needs_docs:
        steps.append("→ needs_web only → web_search_rag")
    elif c.needs_docs and c.needs_web:
        steps.append("→ needs_docs AND needs_web → hybrid_rag")
    elif c.needs_graph:
        steps.append("→ needs_graph → graph_rag")
    else:
        steps.append("→ default → self_rag")
    
    final = selector.select(c)
    steps.append(f"\n★ Result: {final}")
    
    print(f"Query: {query}")
    print("─" * 40)
    print("\n".join(steps))
    print()

trace_selection("Latest breakthroughs in quantum computing")
trace_selection("What is the capital of Japan?")
trace_selection("Compare deep learning and reinforcement learning with recent research papers")

## 4. Strategy → Implementation Map

| Strategy Name | Class | Module |
|---|---|---|
| `direct_llm` | `DirectLLMStrategy` | `strategies/direct_llm_strategy.py` |
| `document_rag` | `DocumentRAGStrategy` | `strategies/document_rag_strategy.py` |
| `web_search_rag` | `WebSearchStrategy` | `strategies/web_search_strategy.py` |
| `hybrid_rag` | `HybridStrategy` | `strategies/hybrid_strategy.py` |
| `graph_rag` | `GraphRAGStrategy` | `strategies/graph_rag_strategy.py` |
| `self_rag` | `SelfRAGStrategy` | `strategies/self_rag_strategy.py` |

> **Next:** [04 — Document RAG](./04_Document_RAG.ipynb) — deep dive into document retrieval